In [13]:
import argparse
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC


RANDOM_STATE = 42
TARGET = "Revenue"
ROOT = Path.cwd()
MODEL_DIR = ROOT / "models"
ARTIFACT_DIR = ROOT / "artifacts"
DATA_DIR = ROOT / "data"


In [14]:
def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    """Build preprocessing based on the actual feature dtypes."""
    categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()

    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_cols),
            ("categorical", categorical_pipeline, categorical_cols),
        ],
        remainder="drop",
        # Dense output is required by GaussianNB.
        sparse_threshold=0.0,
    )


In [15]:
def build_models():
    """Return the five required classifiers."""
    return {
        "Logistic Regression": LogisticRegression(
            max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "Decision Tree": DecisionTreeClassifier(
            max_depth=8, min_samples_split=10, min_samples_leaf=5,
            class_weight="balanced", random_state=RANDOM_STATE
        ),
        "K-Nearest Neighbors": KNeighborsClassifier(
            n_neighbors=7, weights="distance", metric="minkowski"
        ),
        "Gaussian Naive Bayes": GaussianNB(),
        "Support Vector Machine": SVC(
            kernel="rbf", C=1.0, gamma="scale", probability=True,
            class_weight="balanced", random_state=RANDOM_STATE
        ),
    }


In [16]:
def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    return {
        "Accuracy": accuracy_score(y_test, predictions),
        "AUC": roc_auc_score(y_test, probabilities),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "MCC": matthews_corrcoef(y_test, predictions),
    }, predictions, probabilities


In [17]:
def main(data_path: str):
    data_file = Path(data_path)
    if not data_file.is_absolute():
        data_file = ROOT / data_file
    if not data_file.exists():
        raise FileNotFoundError(
            f"Dataset not found: {data_file}\n"
            "Download the UCI CSV and place it at data/online_shoppers_intention.csv "
            "or pass its path with --data."
        )

    df = pd.read_csv(data_file)
    if TARGET not in df.columns:
        raise ValueError(f"Expected target column '{TARGET}'. Found: {df.columns.tolist()}")

    # Drop exact duplicate rows, if any, before splitting.
    original_rows = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Rows loaded: {original_rows}; rows after duplicate removal: {len(df)}")

    # UCI stores Revenue as boolean. Convert it consistently to 0=no purchase, 1=purchase.
    target_values = df[TARGET]
    if target_values.dtype == bool:
        y = target_values.astype(int)
    else:
        normalized = target_values.astype(str).str.strip().str.lower()
        mapping = {
            "true": 1, "false": 0, "1": 1, "0": 0,
            "yes": 1, "no": 0
        }
        y = normalized.map(mapping)
        if y.isna().any():
            raise ValueError("Revenue must contain boolean/0-1/yes-no values only.")

    X = df.drop(columns=[TARGET]).copy()
    if X.shape[1] < 12 or len(df) < 500:
        raise ValueError(
            f"Dataset does not meet assignment minimums: {X.shape[1]} features, {len(df)} rows."
        )

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )

    # Keep the test data in its original feature representation for Streamlit upload.
    test_output = X_test.copy()
    test_output[TARGET] = y_test.to_numpy()
    test_output.to_csv(ROOT / "test_data.csv", index=False)

    metrics_rows = []
    detailed_results = {}

    for display_name, classifier in build_models().items():
        print(f"\nTraining: {display_name}")
        pipeline = Pipeline(steps=[
            ("preprocessor", build_preprocessor(X_train)),
            ("classifier", classifier),
        ])
        pipeline.fit(X_train, y_train)

        metric_values, predictions, probabilities = evaluate_model(
            pipeline, X_test, y_test
        )
        metrics_rows.append({"Model": display_name, **metric_values})
        print(metrics_rows)

        detailed_results[display_name] = {
            "confusion_matrix": confusion_matrix(
                y_test, predictions, labels=[0, 1]
            ).tolist(),
            "classification_report": classification_report(
                y_test, predictions, labels=[0, 1],
                target_names=["No purchase", "Purchase"],
                output_dict=True, zero_division=0
            ),
        }
        print(pd.Series(metric_values).round(4).to_string())
        print(detailed_results)



In [18]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--data", default="/Users/mradulmehrotra/Documents/git/ml-assignment-2/data/online_shoppers_intention.csv",
        help="Path to the downloaded UCI CSV file."
    )
    args = parser.parse_args(args=[])
    main(args.data)


Rows loaded: 12330; rows after duplicate removal: 12205

Training: Logistic Regression
[{'Model': 'Logistic Regression', 'Accuracy': 0.8512904547316673, 'AUC': np.float64(0.9109019526075027), 'Precision': 0.5163511187607573, 'Recall': 0.7853403141361257, 'F1': 0.6230529595015576, 'MCC': np.float64(0.5535674350544707)}]
Accuracy     0.8513
AUC          0.9109
Precision    0.5164
Recall       0.7853
F1           0.6231
MCC          0.5536
{'Logistic Regression': {'confusion_matrix': [[1778, 281], [82, 300]], 'classification_report': {'No purchase': {'precision': 0.9559139784946237, 'recall': 0.8635259834871297, 'f1-score': 0.907374330186272, 'support': 2059.0}, 'Purchase': {'precision': 0.5163511187607573, 'recall': 0.7853403141361257, 'f1-score': 0.6230529595015576, 'support': 382.0}, 'accuracy': 0.8512904547316673, 'macro avg': {'precision': 0.7361325486276905, 'recall': 0.8244331488116277, 'f1-score': 0.7652136448439149, 'support': 2441.0}, 'weighted avg': {'precision': 0.887125362182

/Users/mradulmehrotra/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/mradulmehrotra/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/mradulmehrotra/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/mradulmehrotra/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/mradulmehrotra/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:

[{'Model': 'Logistic Regression', 'Accuracy': 0.8512904547316673, 'AUC': np.float64(0.9109019526075027), 'Precision': 0.5163511187607573, 'Recall': 0.7853403141361257, 'F1': 0.6230529595015576, 'MCC': np.float64(0.5535674350544707)}, {'Model': 'Decision Tree', 'Accuracy': 0.8594838181073331, 'AUC': np.float64(0.8927114773857081), 'Precision': 0.5348837209302325, 'Recall': 0.7827225130890052, 'F1': 0.6354941551540914, 'MCC': np.float64(0.5676023971479615)}, {'Model': 'K-Nearest Neighbors', 'Accuracy': 0.8803768947152806, 'AUC': np.float64(0.8239207768728276), 'Precision': 0.7045454545454546, 'Recall': 0.40575916230366493, 'F1': 0.5149501661129569, 'MCC': np.float64(0.4747521305268141)}]
Accuracy     0.8804
AUC          0.8239
Precision    0.7045
Recall       0.4058
F1           0.5150
MCC          0.4748
{'Logistic Regression': {'confusion_matrix': [[1778, 281], [82, 300]], 'classification_report': {'No purchase': {'precision': 0.9559139784946237, 'recall': 0.8635259834871297, 'f1-score